In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import split, col
from pyspark.sql.functions import regexp_replace
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [0]:
dbutils.widgets.text("catalogo", "catalog_starcraft")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_ranks = spark.table(f"{catalogo}.{esquema_source}.ranks")

In [0]:
# elimina filas con datos vacíos
df_clean_ranks = df_ranks.dropna()

In [0]:
# Reemplaza los valores nulos por 0
df_filled = df_clean_ranks.fillna(0)


In [0]:
# Elimina filas repetidas
df_no_duplicates = df_filled.dropDuplicates()


In [0]:
display(df_no_duplicates)

In [0]:
# Diccionario de mapeo
nicknames = {
    "이제동": "Jaedong",
    "김정우": "EffOrt",
    "유영진": "Reality",
    "박상현": "Shine",
    "김성대": "Calm",
    "장윤철": "Snow",
    "박성균": "Mind",
    "이재호": "Light",
    "김지성": "Bogus",
    "윤수철": "Movie",
    "김명운": "Zero",
    "송병구": "Stork",
    "도재욱": "Best",
    "최호선": "Kwanro",
    "김택용": "Bisu",
    "조기석": "Sea",
    "김경모": "Modesty",
    "원선재": "SkyHigh",
    "황병영": "Leta",
    "임홍규": "Larva",
    "김민철": "Soulkey",
    "어윤수": "soO",
    "임진묵": "Mong",
    "지동원": "Piano",
    "김윤중": "free",
    "조일장": "hero",
    "김윤환": "Shuttle",
    "박수범": "Hyuk",
    "정영재": "Rush",
    "김범수": "Tyson",
    "윤찬희": "Britney",
    "이영호": "Flash",
    "박현재": "HiyA",
    "김학수": "Horang2",
    "방태수": "815",
    "진영화": "Pusan",
    "김태영": "Really",
    "홍덕": "Jaehoon",
    "김재현": "Sacsri",
    "신상문": "Midas",
    "이예훈": "Hyun",
    "김성민": "Saint",
    "이창우": "Hydra",
    "변현제": "Mini",
    "손찬웅": "BeStGod"
}

# UDF para traducir nombres
def translate_name(name):
    # Algunos nombres vienen con inicial de raza (ej: "이제동 Z"), separamos
    base_name = name.split()[0]
    return nicknames.get(base_name, base_name)

translate_udf = udf(translate_name, StringType())

# Supongamos que tu DataFrame se llama df y tiene la columna "Player"
df_translated = df_no_duplicates.withColumn("Player_Nickname", translate_udf(df_no_duplicates["player"]))


In [0]:
display(df_translated)

In [0]:
# Crear una columna de nombre "raza" extraida de la ultima letra de la columna "Player"
df_split = df_translated.withColumn("raza", split(col("player"), " ").getItem(1))
display(df_split)


In [0]:
# separar en diferentes columnas aquellas que contienen valores separados por comas
df_split = df_split.withColumn("vs_zerg_win", split(col("vs_zerg"), ",").getItem(0)) \
             .withColumn("vs_zerg_los", split(col("vs_zerg"), ",").getItem(1)) \
             .withColumn("vs_protoss_win", split(col("vs_protoss"), ",").getItem(0)) \
             .withColumn("vs_protoss_los", split(col("vs_protoss"), ",").getItem(1)) \
             .withColumn("vs_terran_win", split(col("vs_terran"), ",").getItem(0)) \
             .withColumn("vs_terran_los", split(col("vs_terran"), ",").getItem(1))

In [0]:
display(df_split)

In [0]:
# Eliminar columnas
df_split = df_split.drop("vs_zerg", "vs_protoss", "vs_terran")

In [0]:
# Eliminar los valores string en las columas que tienen numeros y columnas

df_clean_split = df_split.withColumn("vs_zerg_win", regexp_replace(col("vs_zerg_win"), "[^0-9]", "").cast("int")) \
                   .withColumn("vs_zerg_los", regexp_replace(col("vs_zerg_los"), "[^0-9]", "").cast("int")) \
                   .withColumn("vs_protoss_win", regexp_replace(col("vs_protoss_win"), "[^0-9]", "").cast("int")) \
                   .withColumn("vs_protoss_los", regexp_replace(col("vs_protoss_los"), "[^0-9]", "").cast("int")) \
                   .withColumn("vs_terran_win", regexp_replace(col("vs_terran_win"), "[^0-9]", "").cast("int")) \
                   .withColumn("vs_terran_los", regexp_replace(col("vs_terran_los"), "[^0-9]", "").cast("int"))

In [0]:
df_clean_split.show()

In [0]:
# Crear un nuevo dataframe a partir de df_clean_split, sin la columna "player", y renombrar la columna "Player_Nickname" a "player"
df_players = df_clean_split.drop("player").withColumnRenamed("Player_Nickname", "player")
df_players.show()

### Guardar Tabla Players_stats

In [0]:
# borrar la table "players_stats" si existe
spark.sql(f"DROP TABLE IF EXISTS {catalogo}.{esquema_sink}.players_stats")

In [0]:
df_players.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{esquema_sink}.players_stats')